In [ ]:
!pip install -q fastapi uvicorn peft transformers accelerate pillow pyngrok nest_asyncio bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
LORA_PATH = "/content/drive/MyDrive/Colab Notebooks/VU_DL_Team_Project/outputs/v3_mild_oversample/final"
print(os.listdir(LORA_PATH))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']


In [ ]:
import shutil, os

os.makedirs('/content/addin/taskpane', exist_ok=True)

shutil.copy('/content/drive/MyDrive/TeamTask/addin/taskpane/taskpane.html', '/content/addin/taskpane/taskpane.html')
shutil.copy('/content/drive/MyDrive/TeamTask/addin/taskpane/taskpane.js', '/content/addin/taskpane/taskpane.js')
shutil.copy('/content/drive/MyDrive/TeamTask/addin/manifest.xml', '/content/addin/manifest.xml')

print(os.listdir('/content/addin/taskpane'))

['taskpane.js', 'taskpane.html']


In [ ]:
from huggingface_hub import login
login(token="HUGGING_FACE_TOKEN")

app.py

In [ ]:
with open('/content/app.py', 'w') as f:
    f.write("""import base64, io, re, os, uuid
from typing import Optional
import torch
from fastapi import FastAPI, HTTPException, Header
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from PIL import Image, ImageFilter
from pydantic import BaseModel
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
from transformers import BitsAndBytesConfig

LORA_PATH   = "/content/drive/MyDrive/Colab Notebooks/VU_DL_Team_Project/outputs/v3_mild_oversample/final"
MODEL_ID    = "google/paligemma2-3b-pt-448"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
PROMPT      = "detect sensitive\\n"
BLUR_RADIUS = 20
SECRET_KEY  = "demo-secret-2024"

print(f"Loading on {DEVICE}...")
processor = AutoProcessor.from_pretrained(LORA_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()
print("Model ready!")

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

sessions = {}
edit_results = {}

class BlurRequest(BaseModel):
    image_b64: str
    filename: Optional[str] = "attachment.png"

class BoundingBox(BaseModel):
    x1: int; y1: int; x2: int; y2: int
    label: str

class BlurResponse(BaseModel):
    blurred_image_b64: str
    boxes: list[BoundingBox]
    original_width: int
    original_height: int

class EditorConfirmRequest(BaseModel):
    session_id: str
    boxes: list[BoundingBox]

class EditorConfirmResponse(BaseModel):
    blurred_image_b64: str

def load_image(b64):
    if "," in b64:
        b64 = b64.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")

def run_inference(image):
    inputs = processor(text=PROMPT, images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    prompt_len = inputs["input_ids"].shape[1]
    return processor.decode(out[0][prompt_len:], skip_special_tokens=False)

def parse_boxes(text, w, h):
    boxes = []
    pattern = re.compile(r"<loc(\\d{4})><loc(\\d{4})><loc(\\d{4})><loc(\\d{4})>\\s*(\\w+)?")
    for m in pattern.finditer(text):
        ny1, nx1, ny2, nx2 = (int(m.group(i)) for i in range(1, 5))
        label = m.group(5) or "sensitive"
        boxes.append(BoundingBox(
            x1=int(nx1/1024*w), y1=int(ny1/1024*h),
            x2=int(nx2/1024*w), y2=int(ny2/1024*h),
            label=label,
        ))
    return boxes

def apply_blur(image, boxes):
    result = image.copy()
    for b in boxes:
        region = result.crop((b.x1, b.y1, b.x2, b.y2))
        result.paste(region.filter(ImageFilter.GaussianBlur(BLUR_RADIUS)), (b.x1, b.y1))
    return result

def to_b64(image):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

@app.post("/process-image", response_model=BlurResponse)
async def process_image(req: BlurRequest, x_api_key: str = Header(...)):
    if x_api_key != SECRET_KEY:
        raise HTTPException(403, "Forbidden")
    try:
        image = load_image(req.image_b64)
    except Exception as e:
        raise HTTPException(400, f"Invalid image: {e}")
    w, h = image.size
    raw   = run_inference(image)
    print("Model output:", raw)
    boxes = parse_boxes(raw, w, h)
    return BlurResponse(
        blurred_image_b64=to_b64(apply_blur(image, boxes) if boxes else image),
        boxes=boxes,
        original_width=w,
        original_height=h,
    )

@app.post("/create-session")
async def create_session(req: BlurRequest, x_api_key: str = Header(...)):
    if x_api_key != SECRET_KEY:
        raise HTTPException(403, "Forbidden")
    try:
        image = load_image(req.image_b64)
    except Exception as e:
        raise HTTPException(400, f"Invalid image: {e}")
    session_id = str(uuid.uuid4())
    sessions[session_id] = {
        "image_b64": req.image_b64,
        "width": image.width,
        "height": image.height,
    }
    return {"session_id": session_id}

@app.get("/session/{session_id}")
async def get_session(session_id: str):
    if session_id not in sessions:
        raise HTTPException(404, "Session not found")
    s = sessions[session_id]
    return {"image_b64": s["image_b64"], "width": s["width"], "height": s["height"]}

@app.post("/confirm-edit", response_model=EditorConfirmResponse)
async def confirm_edit(req: EditorConfirmRequest):
    if req.session_id not in sessions:
        raise HTTPException(404, "Session not found")
    s = sessions[req.session_id]
    image = load_image(s["image_b64"])
    blurred = apply_blur(image, req.boxes)
    blurred_b64 = to_b64(blurred)
    edit_results[req.session_id] = blurred_b64
    del sessions[req.session_id]
    return EditorConfirmResponse(blurred_image_b64=blurred_b64)

@app.get("/edit-result/{session_id}")
async def get_edit_result(session_id: str):
    if session_id not in edit_results:
        return {"ready": False}
    return {"ready": True, "blurred_image_b64": edit_results.pop(session_id)}

@app.get("/health")
async def health():
    return {"status": "ok", "device": DEVICE}

app.mount("/addin", StaticFiles(directory="/content/addin"), name="addin")
""")

app.py written!


URL generation

In [ ]:
import nest_asyncio, uvicorn, threading, time, requests
from pyngrok import ngrok

nest_asyncio.apply()
ngrok.set_auth_token("NGROK_TOKEN")

def run():
    uvicorn.run("app:app", host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=run, daemon=True).start()
time.sleep(10)

tunnel = ngrok.connect(8000)
print("✅ Backend URL:", tunnel.public_url)

def keepalive():
    while True:
        try:
            requests.get(f"{tunnel.public_url}/health",
                headers={"ngrok-skip-browser-warning": "true"}, timeout=5)
        except:
            pass
        time.sleep(30)

threading.Thread(target=keepalive, daemon=True).start()

✅ Backend URL: https://grain-scheming-ellipse.ngrok-free.dev
✅ Keepalive started!


In [ ]:
import requests
r = requests.get("https://grain-scheming-ellipse.ngrok-free.dev/health",
    headers={"ngrok-skip-browser-warning": "true"})
print(r.status_code, r.json())

INFO:     35.240.159.148:0 - "GET /health HTTP/1.1" 200 OK
200 {'status': 'ok', 'device': 'cuda'}


Edit page

In [ ]:
with open('/content/addin/editor.html', 'w') as f:
    f.write("""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Privacy Masker - Edit Boxes</title>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: "Segoe UI", sans-serif; background: #f5f5f5; padding: 16px; }
    h1 { font-size: 18px; margin-bottom: 4px; }
    .subtitle { font-size: 13px; color: #666; margin-bottom: 16px; }
    .canvas-container { position: relative; display: inline-block; border: 1px solid #ddd; background: #fff; cursor: crosshair; }
    canvas { display: block; }
    .btn-row { margin-top: 12px; display: flex; gap: 8px; }
    .btn { padding: 8px 16px; border: none; border-radius: 4px; font-size: 13px; cursor: pointer; font-weight: 500; }
    .btn-primary { background: #1a6fb5; color: #fff; }
    .btn-danger { background: #c4262e; color: #fff; }
    .btn-secondary { background: #e0e0e0; color: #333; }
    #status { margin-top: 12px; font-size: 13px; color: #666; }
    .instructions { font-size: 12px; color: #888; margin-bottom: 8px; }
  </style>
</head>
<body>
  <h1>🔒 Privacy Masker — Edit Boxes</h1>
  <p class="subtitle">Draw additional boxes over sensitive areas, then confirm.</p>
  <p class="instructions">Click and drag to draw a box. Click a box to remove it.</p>

  <div class="canvas-container" id="container">
    <canvas id="canvas"></canvas>
  </div>

  <div class="btn-row">
    <button class="btn btn-primary" id="btn-confirm">✓ Confirm & blur</button>
    <button class="btn btn-secondary" id="btn-clear">Clear all boxes</button>
    <button class="btn btn-danger" id="btn-cancel">✕ Cancel</button>
  </div>

  <div id="status"></div>

  <script>
    const BACKEND_URL = "https://grain-scheming-ellipse.ngrok-free.dev";
    const params = new URLSearchParams(window.location.search);
    const sessionId = params.get("session");
    const callbackUrl = params.get("callback");

    const canvas = document.getElementById("canvas");
    const ctx = canvas.getContext("2d");
    let img = new Image();
    let boxes = [];
    let drawing = false;
    let startX, startY;
    let currentBox = null;
    let scale = 1;

    // Load session data
    fetch(`${BACKEND_URL}/session/${sessionId}`, {
      headers: { "ngrok-skip-browser-warning": "true" }
    })
    .then(r => r.json())
    .then(data => {
      img.onload = () => {
        // Scale to fit screen
        const maxW = Math.min(window.innerWidth - 40, 800);
        scale = maxW / img.naturalWidth;
        canvas.width  = img.naturalWidth  * scale;
        canvas.height = img.naturalHeight * scale;
        redraw();
      };
      img.src = "data:image/png;base64," + data.image_b64;
    })
    .catch(() => {
      document.getElementById("status").textContent = "Error loading session. Please try again.";
    });

    function redraw() {
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
      boxes.forEach((b, i) => {
        ctx.strokeStyle = "#ff6600";
        ctx.lineWidth = 2;
        ctx.strokeRect(b.x1 * scale, b.y1 * scale, (b.x2 - b.x1) * scale, (b.y2 - b.y1) * scale);
        ctx.fillStyle = "rgba(255,102,0,0.15)";
        ctx.fillRect(b.x1 * scale, b.y1 * scale, (b.x2 - b.x1) * scale, (b.y2 - b.y1) * scale);
        ctx.fillStyle = "rgba(255,102,0,0.85)";
        ctx.font = "11px sans-serif";
        ctx.fillText(`Box ${i+1}`, b.x1 * scale + 4, b.y1 * scale + 13);
      });
      if (currentBox) {
        ctx.strokeStyle = "#1a6fb5";
        ctx.lineWidth = 2;
        ctx.setLineDash([4, 4]);
        ctx.strokeRect(currentBox.x1, currentBox.y1, currentBox.x2 - currentBox.x1, currentBox.y2 - currentBox.y1);
        ctx.setLineDash([]);
      }
    }

    canvas.addEventListener("mousedown", e => {
      const rect = canvas.getBoundingClientRect();
      startX = e.clientX - rect.left;
      startY = e.clientY - rect.top;
      drawing = true;
    });

    canvas.addEventListener("mousemove", e => {
      if (!drawing) return;
      const rect = canvas.getBoundingClientRect();
      const x = e.clientX - rect.left;
      const y = e.clientY - rect.top;
      currentBox = { x1: Math.min(startX, x), y1: Math.min(startY, y), x2: Math.max(startX, x), y2: Math.max(startY, y) };
      redraw();
    });

    canvas.addEventListener("mouseup", e => {
      if (!drawing) return;
      drawing = false;
      if (currentBox && (currentBox.x2 - currentBox.x1) > 5 && (currentBox.y2 - currentBox.y1) > 5) {
        boxes.push({
          x1: Math.round(currentBox.x1 / scale),
          y1: Math.round(currentBox.y1 / scale),
          x2: Math.round(currentBox.x2 / scale),
          y2: Math.round(currentBox.y2 / scale),
          label: "sensitive"
        });
      }
      currentBox = null;
      redraw();
    });

    canvas.addEventListener("click", e => {
      if (drawing) return;
      const rect = canvas.getBoundingClientRect();
      const x = (e.clientX - rect.left) / scale;
      const y = (e.clientY - rect.top)  / scale;
      boxes = boxes.filter(b => !(x >= b.x1 && x <= b.x2 && y >= b.y1 && y <= b.y2));
      redraw();
    });

    document.getElementById("btn-clear").addEventListener("click", () => {
      boxes = [];
      redraw();
    });

    document.getElementById("btn-confirm").addEventListener("click", async () => {
      if (boxes.length === 0) {
        document.getElementById("status").textContent = "Draw at least one box first!";
        return;
      }
      document.getElementById("status").textContent = "Processing...";
      document.getElementById("btn-confirm").disabled = true;

      const resp = await fetch(`${BACKEND_URL}/confirm-edit`, {
        method: "POST",
        headers: {
          "Content-Type": "application/json",
          "ngrok-skip-browser-warning": "true"
        },
        body: JSON.stringify({ session_id: sessionId, boxes })
      });
      const data = await resp.json();

      // Notify the bot via callback
      if (callbackUrl) {
        await fetch(callbackUrl, {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify({ blurred_image_b64: data.blurred_image_b64 })
        });
      }

      document.getElementById("status").textContent = "✅ Done! You can close this tab.";
    });

    document.getElementById("btn-cancel").addEventListener("click", () => {
      document.getElementById("status").textContent = "Cancelled. You can close this tab.";
    });
  </script>
</body>
</html>
""")
print("editor.html created!")

editor.html created!


Slack Bot

In [ ]:
with open('/content/slack_bot.py', 'w') as f:
    f.write("""import os, requests, base64, threading, time
from slack_bolt import App
from slack_bolt.adapter.socket_mode import SocketModeHandler

SLACK_BOT_TOKEN = "SLACK_BOT_TOKEN"
SLACK_APP_TOKEN = "SLACK_APP_TOKEN"
BACKEND_URL     = "https://grain-scheming-ellipse.ngrok-free.dev"
SECRET_KEY      = "demo-secret-2024"

app = App(token=SLACK_BOT_TOKEN)

pending = {}

def get_channels(client):
    result = client.conversations_list(types="public_channel")
    return {ch["name"]: ch["id"] for ch in result["channels"]}

def get_username(client, user_id):
    try:
        info = client.users_info(user=user_id)
        return info["user"]["real_name"] or info["user"]["name"]
    except:
        return "Unknown"

def build_channel_options(client):
    channels = get_channels(client)
    return [
        {
            "text": {"type": "plain_text", "text": f"#{name}"},
            "value": ch_id
        }
        for name, ch_id in channels.items()
    ]

def post_blurred(client, say, file_id, channel_id):
    data = pending.pop(file_id, None)
    if not data:
        say("Session expired, please re-upload the image.")
        return
    img_bytes = base64.b64decode(data["blurred_b64"])
    message_text = data.get("message_text", "")
    user_id = data.get("user_id", "")
    username = get_username(client, user_id)
    comment = f"Shared by @{username}\\n{message_text}" if message_text else f"Shared by @{username}"
    client.files_upload_v2(
        channel=channel_id,
        content=img_bytes,
        filename=data["filename"],
        initial_comment=comment
    )
    say("✅ Done!")

@app.event("message")
def handle_message(event, client, say):
    if event.get("subtype") != "file_share":
        return

    channel = event.get("channel", "")
    files   = event.get("files", [])
    message_text = event.get("text", "")
    user_id = event.get("user", "")

    if not files:
        return

    channel_info = client.conversations_info(channel=channel)
    if not channel_info["channel"].get("is_im", False):
        say("👋 Please send images directly to me in a DM to keep them private!")
        return

    for file_info in files:
        if file_info["mimetype"] not in ["image/png","image/jpeg","image/gif","image/webp"]:
            say("Please send an image file (PNG, JPEG, GIF or WEBP).")
            return

        url = file_info["url_private_download"]
        img_bytes = requests.get(url, headers={"Authorization": f"Bearer {SLACK_BOT_TOKEN}"}).content
        b64 = base64.b64encode(img_bytes).decode()

        say("🔍 Scanning for sensitive information...")

        resp = requests.post(
            f"{BACKEND_URL}/process-image",
            headers={"x-api-key": SECRET_KEY, "Content-Type": "application/json", "ngrok-skip-browser-warning": "true"},
            json={"image_b64": b64, "filename": file_info["name"]}
        )
        data = resp.json()
        boxes = data["boxes"]

        if not boxes:
            say("✅ No sensitive information detected. Safe to share!")
            return

        session_resp = requests.post(
            f"{BACKEND_URL}/create-session",
            headers={"x-api-key": SECRET_KEY, "Content-Type": "application/json", "ngrok-skip-browser-warning": "true"},
            json={"image_b64": b64, "filename": file_info["name"]}
        )
        session_id = session_resp.json()["session_id"]
        editor_url = f"{BACKEND_URL}/addin/editor.html?session={session_id}"

        file_id = file_info["id"]
        pending[file_id] = {
            "blurred_b64": data["blurred_image_b64"],
            "filename": file_info["name"],
            "dm_channel": channel,
            "message_text": message_text,
            "user_id": user_id,
            "session_id": session_id,
            "editor_url": editor_url
        }

        # Upload blurred preview to DM
        blurred_bytes = base64.b64decode(data["blurred_image_b64"])
        client.files_upload_v2(
            channel=channel,
            content=blurred_bytes,
            filename="preview_" + file_info["name"],
            initial_comment="👆 Preview of blurred image:"
        )

        box_list = "\\n".join([f"• {b['label']} at ({b['x1']},{b['y1']}) → ({b['x2']},{b['y2']})" for b in boxes])
        opts = build_channel_options(client)

        say(
            blocks=[
                {
                    "type": "section",
                    "text": {"type": "mrkdwn", "text": f"⚠️ *{len(boxes)} sensitive region(s) detected:*\\n{box_list}\\n\\nWhat would you like to do?"}
                },
                {
                    "type": "actions",
                    "elements": [
                        {
                            "type": "static_select",
                            "placeholder": {"type": "plain_text", "text": "Post blurred to channel..."},
                            "options": opts,
                            "action_id": "select_channel"
                        },
                        {
                            "type": "button",
                            "text": {"type": "plain_text", "text": "✏️ Edit boxes"},
                            "action_id": "edit_boxes",
                            "value": file_id
                        },
                        {
                            "type": "button",
                            "text": {"type": "plain_text", "text": "✕ Discard"},
                            "style": "danger",
                            "action_id": "decline_blur",
                            "value": file_id
                        }
                    ]
                }
            ],
            text="Sensitive info detected"
        )

@app.action("select_channel")
def handle_channel_select(ack, body, client, say):
    ack()
    channel_id = body["actions"][0]["selected_option"]["value"]
    file_id = next(iter(pending), None)
    if not file_id:
        say("Session expired, please re-upload the image.")
        return
    post_blurred(client, say, file_id, channel_id)

@app.action("edit_boxes")
def handle_edit_boxes(ack, body, client, say):
    ack()
    file_id = body["actions"][0]["value"]
    if file_id not in pending:
        say("Session expired, please re-upload the image.")
        return

    opts = build_channel_options(client)

    say(
        blocks=[
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": "✏️ Select a channel to post to after editing:"}
            },
            {
                "type": "actions",
                "elements": [
                    {
                        "type": "static_select",
                        "placeholder": {"type": "plain_text", "text": "Select channel..."},
                        "options": [
                            {
                                "text": o["text"],
                                "value": f"{o['value']}|{file_id}"
                            }
                            for o in opts
                        ],
                        "action_id": "select_channel_after_edit"
                    }
                ]
            }
        ],
        text="Select channel"
    )

@app.action("select_channel_after_edit")
def handle_channel_after_edit(ack, body, client, say):
    ack()
    value = body["actions"][0]["selected_option"]["value"]
    channel_id, file_id = value.split("|")

    if file_id not in pending:
        say("Session expired, please re-upload the image.")
        return

    session_id = pending[file_id]["session_id"]
    editor_url = pending[file_id]["editor_url"]
    say(f"🔗 Open the editor and draw additional boxes:\\n{editor_url}")

    def poll(session_id, file_id, channel_id):
        for _ in range(60):
            time.sleep(3)
            r = requests.get(
                f"{BACKEND_URL}/edit-result/{session_id}",
                headers={"ngrok-skip-browser-warning": "true"}
            )
            result = r.json()
            if result.get("ready"):
                if file_id in pending:
                    pending[file_id]["blurred_b64"] = result["blurred_image_b64"]
                    post_blurred(client, say, file_id, channel_id)
                return
        say("⏰ Editor timed out. Please try again.")

    threading.Thread(target=poll, args=(session_id, file_id, channel_id), daemon=True).start()

@app.action("decline_blur")
def handle_decline(ack, say):
    ack()
    say("❌ Discarded. Image will not be shared.")

@app.event("file_shared")
def handle_file_shared(body, logger):
    pass

if __name__ == "__main__":
    handler = SocketModeHandler(app, SLACK_APP_TOKEN)
    handler.start()
""")
print("slack_bot.py updated!")

slack_bot.py updated!


In [ ]:
!pip install slack-bolt

Run slack bot

In [ ]:
!python3 /content/slack_bot.py

⚡️ Bolt app is running!
INFO:     35.240.159.148:0 - "GET /health HTTP/1.1" 200 OK


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


Model output: <loc0229><loc0102><loc0359><loc0974> sensitive<eos>
INFO:     35.240.159.148:0 - "POST /process-image HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "POST /create-session HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /health HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /health HTTP/1.1" 200 OK


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


Model output: <loc0229><loc0102><loc0359><loc0974> sensitive<eos>
INFO:     35.240.159.148:0 - "POST /process-image HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "POST /create-session HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /health HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /health HTTP/1.1" 200 OK


...

Model output: <loc0229><loc0102><loc0359><loc0974> sensitive<eos>
INFO:     35.240.159.148:0 - "POST /process-image HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "POST /create-session HTTP/1.1" 200 OK
INFO:     78.56.245.245:0 - "GET /addin/editor.html?session=6ed1f6ea-ab37-4f6f-8a69-431bec3dee12 HTTP/1.1" 200 OK
INFO:     54.209.158.81:0 - "GET /addin/editor.html?session=6ed1f6ea-ab37-4f6f-8a69-431bec3dee12 HTTP/1.1" 206 Partial Content
INFO:     78.56.245.245:0 - "GET /session/6ed1f6ea-ab37-4f6f-8a69-431bec3dee12 HTTP/1.1" 200 OK
INFO:     54.221.17.218:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     35.240.159.148:0 - "GET /edit-result/6ed1f6ea-ab37-4f6f-8a69-431bec3dee12 HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /health HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /edit-result/6ed1f6ea-ab37-4f6f-8a69-431bec3dee12 HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /edit-result/6ed1f6ea-ab37-4f6f-8a69-431bec3dee12 HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "

You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


Model output: <loc0229><loc0102><loc0359><loc0974> sensitive<eos>
INFO:     35.240.159.148:0 - "POST /process-image HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /health HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "POST /create-session HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /health HTTP/1.1" 200 OK


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


Model output: <loc0229><loc0102><loc0359><loc0974> sensitive<eos>
INFO:     35.240.159.148:0 - "POST /process-image HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "POST /create-session HTTP/1.1" 200 OK
INFO:     32.197.240.110:0 - "GET /addin/editor.html?session=9fa0087f-baaa-477d-b08d-945dd1770f86 HTTP/1.1" 206 Partial Content
INFO:     54.157.221.221:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     78.56.245.245:0 - "GET /addin/editor.html?session=9fa0087f-baaa-477d-b08d-945dd1770f86 HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /edit-result/9fa0087f-baaa-477d-b08d-945dd1770f86 HTTP/1.1" 200 OK
INFO:     78.56.245.245:0 - "GET /session/9fa0087f-baaa-477d-b08d-945dd1770f86 HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /edit-result/9fa0087f-baaa-477d-b08d-945dd1770f86 HTTP/1.1" 200 OK
INFO:     78.56.245.245:0 - "POST /confirm-edit HTTP/1.1" 200 OK
INFO:     35.240.159.148:0 - "GET /edit-result/9fa0087f-baaa-477d-b08d-945dd1770f86 HTTP/1.1" 200 OK
INFO:     35.240.159.1